## **Setup**

In [1]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

WORKING_DIR = os.getcwd()

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = "/home/luigi/RecSys" if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

Repo already exists — pulling latest changes
Already up to date.


In [2]:
if IS_COLAB or False:  # Set to True if you want to recompile Cython files
    os.chdir(LOCAL_REPO_PATH)
    !python run_compile_all_cython.py
    os.chdir(WORKING_DIR)

In [3]:
if IS_COLAB or IS_KAGGLE:
    !pip install optuna

import optuna

In [4]:
import importlib
import numpy as np

from Challenge import paths
importlib.reload(paths)

from Challenge.hyper_tuning import ModelOptimizer

Running on local — storage at: /home/luigi/RecSys
Running on local — storage at: /home/luigi/RecSys


/home/luigi/RecSys/Challenge/hyper_tuning.py:75: ExperimentalWarning: WilcoxonPruner is experimental (supported from v3.6.0). The interface can change in the future.
  def create_study(self, study_name, direction="maximize", load_if_exists=True, pruner=optuna.pruners.WilcoxonPruner()):
/home/luigi/RecSys/Challenge/hyper_tuning.py:107: ExperimentalWarning: WilcoxonPruner is experimental (supported from v3.6.0). The interface can change in the future.
  def create_and_optimize_study(self, study_name, objective_function, n_trials=50, direction="maximize", load_if_exists=True, pruner=optuna.pruners.WilcoxonPruner()):


## **Load Data**

In [5]:
URM_train, URM_val = paths.load_holdout_split()

## **Load Models**

### **Train Fast Models**

In [ ]:
import json

def get_best_params(json_path: str) -> dict:
    with open(json_path, "r") as f:
        data = json.load(f)
        
        best_study = None
        for study, values in data.items():
            if best_study is None:
                best_study = study
                continue

            if values["best_score"] > best_study["best_score"]:
                best_study = values

    return best_study["best_params"]

In [ ]:
from Recommenders.NonPersonalizedRecommender import TopPop
from Recommenders.KNN.UserKNNCFRecommender import UserKNNCFRecommender
from Recommenders.KNN.ItemKNNCFRecommender import ItemKNNCFRecommender
from Recommenders.GraphBased.P3alphaRecommender import P3alphaRecommender
from Recommenders.EASE_R.EASE_R_Recommender import EASE_R_Recommender

models = {}

# TopPop
model = TopPop(URM_train)
model.fit()
models["TopPop"] = model

# KNN
KNN_similarities = ["cosine", "jaccard", "asymmetric", "tversky", "dice"]

for sim in KNN_similarities:
    path = os.path.join(LOCAL_REPO_PATH, f"performance_logs/KNN_{sim}.json")
    best_params = get_best_params(path)
            
    user_model = UserKNNCFRecommender(URM_train)
    user_model.fit(**best_params)
    models[f"UserKNNCF_{sim}"] = user_model
    
    path = os.path.join(LOCAL_REPO_PATH, f"performance_logs/KNN_{sim}.json")
    best_params = get_best_params(path)

    item_model = ItemKNNCFRecommender(URM_train)
    item_model.fit(**best_params)
    models[f"ItemKNNCF_{sim}"] = item_model

# P3alpha
path = os.path.join(LOCAL_REPO_PATH, f"performance_logs/P3alpha.json")
best_params = get_best_params(path)
p3alpha_model = P3alphaRecommender(URM_train)
p3alpha_model.fit(**best_params)
models["P3alpha"] = p3alpha_model

# EASE-R
path = os.path.join(LOCAL_REPO_PATH, f"performance_logs/EASE_R.json")
best_params = get_best_params(path)
ease_r_model = EASE_R_Recommender(URM_train)
ease_r_model.fit(**best_params)
models["EASE_R"] = ease_r_model

In [ ]:
models.keys()

### **Load Slow Models**

In [ ]:
from Recommenders.SLIM.SLIMElasticNetRecommender import MultiThreadSLIM_SLIMElasticNetRecommender
from Recommenders.SLIM.Cython.SLIM_BPR_Cython import SLIM_BPR_Cython
from Recommenders.MatrixFactorization.IALSRecommender import IALSRecommender
from Recommenders.MatrixFactorization.PureSVDRecommender import PureSVDRecommender
from Recommenders.MatrixFactorization.NMFRecommender import NMFRecommender
from Recommenders.GraphBased.RP3betaRecommender import RP3betaRecommender
from Recommenders.Neural.MultVAERecommender import MultVAERecommender

TRAIN = False
path_models = os.path.join(LOCAL_REPO_PATH, "hybrid_models")

In [ ]:
if TRAIN:
    # SLIM ElasticNet
    path = os.path.join(LOCAL_REPO_PATH, f"performance_logs/SLIM_ElasticNet.json")
    best_params = get_best_params(path)
    slim_en_model = MultiThreadSLIM_SLIMElasticNetRecommender(URM_train)
    slim_en_model.fit(**best_params)
    models["SLIM_ElasticNet"] = slim_en_model

    # SLIM BPR
    path = os.path.join(LOCAL_REPO_PATH, f"performance_logs/SLIM_BPR.json")
    best_params = get_best_params(path)
    slim_bpr_model = SLIM_BPR_Cython(URM_train)
    slim_bpr_model.fit(**best_params)
    models["SLIM_BPR"] = slim_bpr_model

    # IALS
    path = os.path.join(LOCAL_REPO_PATH, f"performance_logs/IALS.json")
    best_params = get_best_params(path)
    ials_model = IALSRecommender(URM_train)
    ials_model.fit(**best_params)
    models["IALS"] = ials_model

    # PureSVD
    path = os.path.join(LOCAL_REPO_PATH, f"performance_logs/PureSVD.json")
    best_params = get_best_params(path)
    pure_svd_model = PureSVDRecommender(URM_train)
    pure_svd_model.fit(**best_params)
    models["PureSVD"] = pure_svd_model

    # RP3beta
    path = os.path.join(LOCAL_REPO_PATH, f"performance_logs/RP3beta.json")
    best_params = get_best_params(path)
    rp3beta_model = RP3betaRecommender(URM_train)
    rp3beta_model.fit(**best_params)
    models["RP3beta"] = rp3beta_model

    # MultVAE
    path = os.path.join(LOCAL_REPO_PATH, f"performance_logs/MultVAE.json")
    best_params = get_best_params(path)
    multvae_model = MultVAERecommender(URM_train)
    multvae_model.fit(**best_params)
    models["MultVAE"] = multvae_model

    # NMF
    path = os.path.join(LOCAL_REPO_PATH, f"performance_logs/NMF.json")
    best_params = get_best_params(path)
    nmf_model = NMFRecommender(URM_train)
    nmf_model.fit(**best_params)
    models["NMF"] = nmf_model

    # Save trained models
    slim_en_model.save_model(os.path.join(path_models, "SLIM_ElasticNet"))
    slim_bpr_model.save_model(os.path.join(path_models, "SLIM_BPR"))
    ials_model.save_model(os.path.join(path_models, "IALS"))
    pure_svd_model.save_model(os.path.join(path_models, "PureSVD"))
    rp3beta_model.save_model(os.path.join(path_models, "RP3beta"))
    multvae_model.save_model(os.path.join(path_models, "MultVAE"))
    nmf_model.save_model(os.path.join(path_models, "NMF"))

else:
    # Load trained models
    slim_en_model = MultiThreadSLIM_SLIMElasticNetRecommender(URM_train)
    slim_en_model.load_model(os.path.join(path_models, "SLIM_ElasticNet"))
    models["SLIM_ElasticNet"] = slim_en_model

    slim_bpr_model = SLIM_BPR_Cython(URM_train)
    slim_bpr_model.load_model(os.path.join(path_models, "SLIM_BPR"))
    models["SLIM_BPR"] = slim_bpr_model

    ials_model = IALSRecommender(URM_train)
    ials_model.load_model(os.path.join(path_models, "IALS"))
    models["IALS"] = ials_model

    pure_svd_model = PureSVDRecommender(URM_train)
    pure_svd_model.load_model(os.path.join(path_models, "PureSVD"))
    models["PureSVD"] = pure_svd_model

    rp3beta_model = RP3betaRecommender(URM_train)
    rp3beta_model.load_model(os.path.join(path_models, "RP3beta"))
    models["RP3beta"] = rp3beta_model

    multvae_model = MultVAERecommender(URM_train)
    multvae_model.load_model(os.path.join(path_models, "MultVAE"))
    models["MultVAE"] = multvae_model

    nmf_model = NMFRecommender(URM_train)
    nmf_model.load_model(os.path.join(path_models, "NMF"))
    models["NMF"] = nmf_model

In [ ]:
models.keys()

## **Check correlation between models prediction**

In [ ]:
import pandas as pd

# Generate prediction with score for each model
model_scores = {}
user_ids = np.arange(URM_train.shape[0])
for model_name, model in models.items():
    print(f"Generating scores for model: {model_name}")
    scores = model._compute_item_score(user_ids)
    
    # Sort scores and keep only top 100
    top_k = 100
    top_k_indices = np.argpartition(-scores, top_k, axis=1)[:, :top_k]
    top_k_scores = np.take_along_axis(scores, top_k_indices, axis=1)

    model_scores[model_name] = (top_k_indices, top_k_scores)

# Create a DataFrame to hold the scores
scores_df = pd.DataFrame(index=user_ids)
for model_name, (indices, scores) in model_scores.items():
    # Create a sparse matrix for the scores
    sparse_scores = sps.csr_matrix((scores.flatten(), 
                                    (np.repeat(user_ids, scores.shape[1]), 
                                     indices.flatten())),
                                   shape=(URM_train.shape[0], URM_train.shape[1]))
    scores_df[model_name] = list(sparse_scores)

scores_df.head()